# Lab 06 — GPU and Kernel Thinking
**Goal:** Benchmark simple matrix operations when a GPU exists and learn to identify compute-bound vs overhead-bound regions.

Free-first: run on CPU or free Colab where possible.

Method: **predict → run → inspect → break → decide**.


In [ ]:
import torch, time
print("torch:",torch.__version__)
print("cuda available:",torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:",torch.cuda.get_device_name(0))


## Predict before running
Predict:
- Why is a 32x32 matrix multiplication often much less efficient than a 2048x2048 one?
- Why can kernel launch overhead matter?
- What changes when work becomes large enough to saturate GPU compute?


In [ ]:
def bench(n=1024, iters=50, device="cpu"):
    a=torch.randn(n,n,device=device)
    b=torch.randn(n,n,device=device)
    if device=="cuda": torch.cuda.synchronize()
    t=time.perf_counter()
    for _ in range(iters):
        c=a@b
    if device=="cuda": torch.cuda.synchronize()
    return (time.perf_counter()-t)/iters
for dev in ["cpu"] + (["cuda"] if torch.cuda.is_available() else []):
    print(dev, bench(1024,20,dev))


## Explain the result
Large matrix operations amortize fixed overhead and expose highly parallel arithmetic. Small operations may be limited by Python/kernel-launch overhead or memory movement instead of raw FLOPs.


In [ ]:
# Change matrix size. Plotting is optional; the key observation is how time scales.
for n in [128,256,512,1024,2048]:
    dev="cuda" if torch.cuda.is_available() else "cpu"
    print(n, bench(n,10,dev))


## Critical-thinking answers
Kernel optimization begins with profiling. Do not optimize the slowest-looking line by intuition alone. First establish whether the workload is compute-, memory-, or communication-bound, then choose the appropriate optimization.

### Sources
https://cs336.stanford.edu/
